In [1]:
from ash import *

ashpath: /Users/tom/Documents/ucl/projects/ash-fork/ash
Sys path: ['/Users/tom/Documents/ucl/projects/ash-fork/ash', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python311.zip', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11/lib-dynload', '', '/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages', '__editable__.ash-0.95.finder.__path_hook__']


/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/pennylane/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                                           ASH                                            
                              A MULTISCALE MODELLING PROGRAM                              
                                        Version: 0.9dev                                        
                               Git commit version: Unknown                                
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
ASH path: /Users/tom/Documents/ucl/projects/ash-fork/ash
Python version: 3.11.12
Python interpreter: /Users/tom/Documents/ucl/projects/ash-fork/.venv/bin/python

ASH Settings after reading defaults and ~/ash_user_settings.ini : 
See https://ash.readthedocs.io/en/latest/basics.html#ash-settings on how to ch

In [2]:
BASIS = 'sto-3g'
XC = 'b3lyp'
CHARGE = 0
MULT = 1

In [3]:
#Defining fragment
frag = Fragment(xyzfile="system_aftersolvent.xyz", charge=CHARGE, mult=MULT)


--------------------------------------------------------------------------------
                                New ASH fragment                                
--------------------------------------------------------------------------------

ASH Fragment creation
Reading coordinates from XYZ file 'system_aftersolvent.xyz' into fragment.
Creating/Updating fragment attributes...
Number of Atoms in fragment: 2637
Formula: O878C1H1758
Label: system_aftersolvent
Charge: 0 Mult: 1

--------------------------------------------------------------------------------


In [4]:
xyz_list = [i for i in zip(frag.elems, frag.coords)]

lines = [str(len(xyz_list)), '']
for symbol, coords in xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

xyz_string = "\n".join(lines)

In [5]:
qm_atoms = [0,1,2,3,4,5]

In [6]:
qm_xyz_list = [
    (frag.elems[i], frag.coords[i])
    for i in qm_atoms
]

lines = [str(len(qm_xyz_list)), '']
for symbol, coords in qm_xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

qm_xyz_string = "\n".join(lines)
print(qm_xyz_string)

6

O 0.0 -1.433 0.0
H -0.962 -1.67 0.0
C 0.0 0.0 0.0
H -0.49 0.417 0.886
H 1.04 0.331 0.0
H -0.49 0.417 -0.886


In [7]:
N_ACT = 2

In [8]:
nbed_theory = NbedTheory(
    geometry=qm_xyz_string,
    n_active_atoms=N_ACT,
    basis=BASIS,
    xc_functional=XC,
    projector='mu',
    localization='spade'
)



                     #####################################                      
                     #                                   #                      
                     #     NbedTheory initialization     #                      
                     #                                   #                      
                     #####################################                      


In [9]:
# water_xml = "/opt/homebrew/Caskroom/miniconda/base/envs/ash-conda/lib/python3.11/site-packages/openmm/app/data/amber14/tip3p.xml"
water_xml = "amber14/tip3p.xml"

openmm_theory = OpenMMTheory(
    xmlfiles=["openff_LIG.xml", water_xml], 
    pdbfile="system_aftersolvent.pdb", 
    periodic=True, 
    autoconstraints=None,
    rigidwater=False
)



                           #########################                            
                           #                       #                            
                           #     OpenMM Theory     #                            
                           #                       #                            
                           #########################                            
OpenMM CPU threads set to: 1
Imported OpenMM library version: 8.3.1

--------------------------------------------------------------------------------
                             Defining OpenMM object                             
--------------------------------------------------------------------------------

Printlevel: 2
No automatic constraints
AutoConstraint setting: None
Rigidwater constraints: False
Hydrogenmass option: 1.5 Da
Using platform: CPU

--------------------------------------------------------------------------------
                            Setting up force fields.

In [10]:
qmmm_theory = QMMMTheory(
    qm_theory = nbed_theory,
    mm_theory = openmm_theory,
    fragment = frag,
    qm_charge = CHARGE,
    qm_mult = MULT,
    qmatoms = qm_atoms,
    printlevel = 3,
)



                            ########################                            
                            #                      #                            
                            #     QM/MM Theory     #                            
                            #                      #                            
                            ########################                            
QM-theory: NbedTheory
MM-theory: OpenMMTheory
All atoms in fragment: 2637
QM region (6 atoms): [0, 1, 2, 3, 4, 5]
MM region (2631 atoms)
QM/MM object selected to use 1 cores
Embedding: elstat
No atomcharges list passed to QMMMTheory object
Getting system charges from OpenMM object
QM-region coordinates (before linkatoms):
   0    O   0.00000000   -1.43300000    0.00000000
   1    H  -0.96200000   -1.67000000    0.00000000
   2    C   0.00000000    0.00000000    0.00000000
   3    H  -0.49000000    0.41700000    0.88600000
   4    H   1.04000000    0.33100000    0.00000000
   5    H  -0.

In [11]:
sp = Singlepoint(
    theory=qmmm_theory, 
    fragment=frag,
)



                        ################################                        
                        #                              #                        
                        #     Singlepoint function     #                        
                        #                              #                        
                        ################################                        
Checking if present in QM/MM object
Found qm_charge and qm_mult attributes.
Using charge=0 and mult=1

Doing single-point Energy job on fragment. Formula: O878C1H1758 Label: system_aftersolvent 
Charge: 0 Mult: 1
  ------------RUNNING QM/MM MODULE------------- 
QM Module: NbedTheory
MM Module: OpenMMTheory
Charge provided from QMMMTheory object:  0
Mult provided from QMMMTheory object:  1
QM-region Charge: 0 Mult: 1
Embedding: Electrostatic
First QMMMTheory run. Running runprep
Inside QMMMTheory runprep

-------------------------------------------------------------
Time to calculate ste

/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/pyscf/dft/libxc.py:512: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


MM elements: ['O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', 'H', 'O', 'H', '

/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:84: RuntimeWarning: divide by zero encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:84: RuntimeWarning: overflow encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:84: RuntimeWarning: invalid value encountered in matmul
  linalg.fractional_matrix_power(ao_overlap, 0.5) @ occupied_orbitals
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:108: RuntimeWarning: divide by zero encountered in matmul
  c_loc_occ = occupied_orbitals @ right_vectors.T
/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages/nbed/localizers/spade.py:108

In [12]:
sp.energy

np.float64(-127.12256308570306)